# Experiment Tracking Example

In [2]:
#| eval: false
#| echo: true
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from tqdm.auto import tqdm

# --- CNN-Modell ---
class FashionCNN(nn.Module):
    def __init__(self, dropout: float = 0.25):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),  # (B, 32, 28, 28)
            nn.ReLU(),
            nn.MaxPool2d(2),                              # (B, 32, 14, 14)
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # (B, 64, 14, 14)
            nn.ReLU(),
            nn.MaxPool2d(2),                              # (B, 64,  7,  7)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# --- Dataloading ---
def get_dataloaders(batch_size: int = 64):
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.2860,), (0.3530,)),
    ])
    train_ds = datasets.FashionMNIST("./data", train=True,  download=True, transform=transform)
    val_ds   = datasets.FashionMNIST("./data", train=False, download=True, transform=transform)
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0),
        DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0),
    )


# --- Trainings-Hilfsfunktionen ---
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for X, y in tqdm(loader):
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * len(X)
    return total_loss / len(loader.dataset)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct = 0.0, 0
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            total_loss += criterion(logits, y).item() * len(X)
            correct += (logits.argmax(dim=1) == y).sum().item()
    n = len(loader.dataset)
    return total_loss / n, correct / n


def train_and_evaluate(lr: float = 1e-3, batch_size: int = 64,
                       dropout: float = 0.25, epochs: int = 10):
    """Trainiert ein FashionCNN und gibt die finale Val-Accuracy zurück."""
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    train_loader, val_loader = get_dataloaders(batch_size)
    model = FashionCNN(dropout=dropout).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    return val_acc  # Rückgabe für HPO-Loops

/Users/johbaum/code/lecture-ki-systeme-code/experiment-tracking/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Define Hyper Parameter and Setup

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameter
params = {
    "epochs": 10,
    "learning_rate": 1e-3,
    "batch_size": 64,
    "dropout": 0.25,
    "optimizer": "Adam",
}

train_loader, val_loader = get_dataloaders(params["batch_size"])
model = FashionCNN(dropout=params["dropout"]).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=params["learning_rate"])

## Train without experiment tracking

Just the plain training loop only experiment tracking and the other improvements to observe the training runs.

> Limited insights and control

In [3]:
for epoch in range(params["epochs"]):
    # --- Training Loop ---
    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)

    # --- Validation ---
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)

    print(f"Epoch: {epoch}: val_loss = {val_loss} val_acc: {val_acc}")

  0%|          | 0/938 [00:00<?, ?it/s]

100%|██████████| 938/938 [00:14<00:00, 64.49it/s]


Epoch: 0: val_loss = 0.32292652616500855 val_acc: 0.8804


100%|██████████| 938/938 [00:15<00:00, 61.59it/s]


Epoch: 1: val_loss = 0.2695245799303055 val_acc: 0.8992


...

Epoch: 2: val_loss = 0.24354146722555162 val_acc: 0.9103


100%|██████████| 938/938 [00:14<00:00, 63.22it/s]


Epoch: 3: val_loss = 0.24955156388282776 val_acc: 0.9137


100%|██████████| 938/938 [00:14<00:00, 64.93it/s]


Epoch: 4: val_loss = 0.24237828751802445 val_acc: 0.918


## Setup Experiment Tracking

NOTE: Make sure mlfow is running.

Start it with

```bash
uv run mlflow server
```

In [5]:
import mlflow
import mlflow.pytorch
import torch

# Experiment und Tracking Server konfigurieren
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("FashionMNIST-CNN")
mlflow.config.enable_system_metrics_logging()

## Train with experiment tracking

The command line output stays the same, but check the mlfow ui by calling [http://127.0.0.1:5000](http://127.0.0.1:5000)

In [5]:


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Hyperparameter
params = {
    "epochs": 10,
    "learning_rate": 1e-4,
    "batch_size": 64,
    "dropout": 0.5,
    "optimizer": "Adam",
}

train_loader, val_loader = get_dataloaders(params["batch_size"])
model = FashionCNN(dropout=params["dropout"]).to(device)
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=params["learning_rate"])

with mlflow.start_run(run_name="cnn-dropout-0.5-lr-1e-4"):
    # (1) Hyperparameter einmalig loggen
    mlflow.log_params(params)

    for epoch in range(params["epochs"]):
        # --- Training Loop ---
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)

        # --- Validation ---
        val_loss, val_acc = evaluate(model, val_loader, criterion, device)

        # (2) Metriken pro Epoche loggen
        mlflow.log_metrics({
            "train_loss":   train_loss,
            "val_loss":     val_loss,
            "val_accuracy": val_acc,
        }, step=epoch)

    # (3) Finales Modell speichern
    mlflow.pytorch.log_model(
        model,
        artifact_path="final_model",
        registered_model_name="FashionMNIST-Classifier",
        input_example=torch.randn(1, 1, 28, 28).numpy(),
    )

2026/04/10 11:56:03 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/04/10 11:56:03 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
100%|██████████| 938/938 [00:15<00:00, 60.78it/s]
2026/04/10 11:58:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/10 11:58:46 INFO mlflow.utils.uv_utils: Detected uv project: found uv.lock and pyproject.toml in /Users/johbaum/code/lecture-ki-systeme-code/experiment-tracking
2026/04/10 11:58:46 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/04/10 11:58:47 INFO mlflow.utils.

🏃 View run cnn-dropout-0.5-lr-1e-4 at: http://127.0.0.1:5000/#/experiments/1/runs/26538b3a3fe74d6b964224d8193c9b30
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


## Register the best model as "champion"

In [6]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

# Besten Run finden
best = mlflow.search_runs(
    experiment_names=["FashionMNIST-CNN"],
    order_by=["metrics.val_accuracy DESC"],
).iloc[0]

run_id = best["run_id"]
model_version = mlflow.register_model(
    f"runs:/{run_id}/final_model",
    name="FashionMNIST-Classifier"
)

client.set_registered_model_alias(
    name="FashionMNIST-Classifier",
    alias="champion",
    version=model_version.version,
)

print(f"Champion: Version {model_version.version}")

Registered model 'FashionMNIST-Classifier' already exists. Creating a new version of this model...
2026/04/10 11:58:48 WARNING mlflow.tracking._model_registry.fluent: Run with id 26538b3a3fe74d6b964224d8193c9b30 has no artifacts at artifact path 'final_model', registering model based on models:/m-598b2062977a423abf67654642298e33 instead
2026/04/10 11:58:48 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: FashionMNIST-Classifier, version 2


Champion: Version 2


Created version '2' of model 'FashionMNIST-Classifier'.


## Use the best Model for inference 

1. Load the model from the artifact store
2. Run the inference

In [ ]:
model = mlflow.pytorch.load_model("models:/FashionMNIST-Classifier@champion")
model.eval()
label_names = ["T-Shirt","Hose","Pullover","Kleid","Mantel",
               "Sandale","Hemd","Sneaker","Tasche","Stiefelette"]

X_test, _ = next(iter(val_loader))
with torch.no_grad():
    preds = model(X_test[:5]).argmax(1)
for i, p in enumerate(preds):
    print(f"Bild {i+1}: {label_names[p.item()]}")
    print(f"Bild {i+1}: Klasse {p.item()} → {label_names[p.item()]}")  



Bild 1: Stiefelette
Bild 1: Klasse 9 → Stiefelette
Bild 2: Pullover
Bild 2: Klasse 2 → Pullover
Bild 3: Hose
Bild 3: Klasse 1 → Hose
Bild 4: Hose
Bild 4: Klasse 1 → Hose
Bild 5: Hemd
Bild 5: Klasse 6 → Hemd


Bild 1: Klasse 8 → Tasche
Bild 2: Klasse 8 → Tasche
Bild 3: Klasse 8 → Tasche
Bild 4: Klasse 8 → Tasche
Bild 5: Klasse 8 → Tasche


## Store the bad predictions

Store the bad predictions for documentation and investigation

In [ ]:
import matplotlib.pyplot as plt
import mlflow
import mlflow.pytorch

def log_worst_predictions(model, loader, criterion, device, n=8):
    model.eval()
    losses, images, preds, labels = [], [], [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            batch_l = nn.functional.cross_entropy(logits, y, reduction='none')
            losses += batch_l.cpu().tolist()
            images += list(X.cpu())
            preds  += logits.argmax(1).cpu().tolist()
            labels += y.cpu().tolist()
    worst = sorted(range(len(losses)), key=lambda i: losses[i], reverse=True)[:n]

    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for ax, idx in zip(axes.flat, worst):
        ax.imshow(images[idx].squeeze(), cmap='gray')
        ax.set_title(
            f"Pred: {label_names[preds[idx]]}\nTrue: {label_names[labels[idx]]}",
            fontsize=8
        )
        ax.axis('off')
    plt.tight_layout()
    mlflow.log_figure(fig, artifact_file="worst_predictions.png")
    plt.close(fig)

with mlflow.start_run(run_name="fehler-analyse"):
    log_worst_predictions(model, val_loader, criterion, device)

2026/04/10 11:58:49 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/04/10 11:58:49 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/04/10 11:58:51 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/04/10 11:58:51 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


🏃 View run fehler-analyse at: http://127.0.0.1:5000/#/experiments/1/runs/d8925655a5954ad3b7f54abbfbab2827
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
